# Semana 4 — Prototipado de Inferencia

Este notebook sirve para prototipar y verificar la lógica de inferencia en producción antes de implementarla en la app web de Streamlit (`prod/app.py` y `prod/utils.py`).

El flujo a verificar es:
1. Cargar el checkpoint unificado (`dev/modelo.pth`).
2. Reconstruir el modelo y cargar los pesos entrenados.
3. Inicializar **MTCNN** para detectar y recortar múltiples rostros.
4. Procesar una imagen de entrada, extraer sus rostros, calcular embeddings y compararlos contra los centroides de clase usando similitud coseno.
5. Aplicar un **umbral de rechazo** para clasificar como "Desconocido" si la similitud es baja.

In [ ]:
# ── 1. Setup e Importaciones ─────────────────────────────────
import sys
import json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from facenet_pytorch import MTCNN, InceptionResnetV1
import matplotlib.pyplot as plt
import matplotlib.patches as patches

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Dispositivo de ejecución:", DEVICE)

In [ ]:
# ── 2. Definición del Modelo ──────────────────────────────────
class JointEmbeddingNet(nn.Module):
    def __init__(self, backbone, embedding_dim=128, num_classes=26):
        super(JointEmbeddingNet, self).__init__()
        self.backbone = backbone
        
        if hasattr(backbone, 'last_linear'):
            # Caso FaceNet (InceptionResnetV1)
            in_features = 512
        elif hasattr(backbone, 'fc'):
            in_features = backbone.fc.in_features
            backbone.fc = nn.Identity()
        elif hasattr(backbone, 'classifier'):
            if isinstance(backbone.classifier, nn.Sequential):
                in_features = backbone.classifier[-1].in_features
                backbone.classifier[-1] = nn.Identity()
            else:
                in_features = backbone.classifier.in_features
                backbone.classifier = nn.Identity()
        else:
            in_features = 512 # Fallback
                
        self.embedding_layer = nn.Linear(in_features, embedding_dim)
        self.classifier_head = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        embeddings = self.embedding_layer(features)
        embeddings = nn.functional.normalize(embeddings, p=2, dim=1)
        logits = self.classifier_head(embeddings)
        return embeddings, logits

In [ ]:
# ── 3. Cargar Checkpoint Unificado ────────────────────────────
CKPT_PATH = Path('modelo.pth')  # Se asume ejecución desde dev/

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"No se encontró el checkpoint en {CKPT_PATH.resolve()}")

print("Cargando checkpoint...")
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)

# Reconstruir arquitectura (se usa FaceNet como backbone preentrenado)
backbone = InceptionResnetV1(pretrained='vggface2')
model = JointEmbeddingNet(backbone, embedding_dim=checkpoint['embedding_dim'], num_classes=checkpoint['num_classes'])

# Cargar los pesos del clasificador entrenado
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()  # Modo evaluación obligatorio

# Extraer centroides y mapeo de clases
centroids = checkpoint['centroids'].to(DEVICE)  # Shape: [26, 128]
idx_to_label = checkpoint['idx_to_label']

print("✓ Checkpoint cargado exitosamente.")
print(f"  - Dimensiones del embedding: {checkpoint['embedding_dim']}")
print(f"  - Cantidad de clases: {checkpoint['num_classes']}")
print(f"  - Centroides en memoria: {centroids.shape}")

In [ ]:
# ── 4. Inicializar Detectores e Inferencia ────────────────────
# keep_all=True permite detectar todos los rostros en fotos grupales
mtcnn = MTCNN(keep_all=True, device=DEVICE)

# Transformación obligatoria e idéntica a la usada en validación/test
preprocess_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✓ MTCNN y transformaciones de preprocesamiento inicializadas.")

In [ ]:
# ── 5. Cargar Imagen de Prueba ────────────────────────────────
# Vamos a buscar una imagen de prueba del test set. 
# Si el dataset no está, podés arrastrar cualquier imagen JPG al directorio dev/ y cambiar la ruta.
import pandas as pd

test_csv = Path('../data/test.csv')
if test_csv.exists():
    test_df = pd.read_csv(test_csv)
    # Tomar una imagen aleatoria de Messi u otro jugador
    sample_row = test_df.sample(n=1, random_state=42).iloc[0]
    img_path = Path('../data/FIFA_2022_ONLY_FACES') / sample_row['image_path']
    print(f"Cargando imagen aleatoria del test set: {img_path} (Real: {sample_row['label']})")
else:
    # Fallback si no está el split: busca la primera imagen que encuentre en la carpeta de caras
    img_files = list(Path('../data/FIFA_2022_ONLY_FACES').rglob('*.jpg'))
    if img_files:
        img_path = img_files[0]
        print(f"Cargando primera imagen encontrada: {img_path}")
    else:
        print("⚠️ No se encontró ninguna imagen en ../data/. Por favor, subí una foto de prueba y asigná 'img_path'.")
        img_path = Path('test_image.jpg') # Cambiar por tu ruta

if img_path.exists():
    img = Image.open(img_path).convert('RGB')
    plt.imshow(img)
    plt.axis('off')
    plt.title("Imagen de prueba")
    plt.show()

In [ ]:
# ── 6. Detección, Recorte y Clasificación con Umbral ───────────
UMBRAL_RECHAZO = 0.65  # Similitud coseno mínima para aceptar la clase

if not img_path.exists():
    print("Por favor crea o sube una imagen de prueba primero.")
else:
    img = Image.open(img_path).convert('RGB')
    
    # 1. Detectar cajas y puntos clave de rostros
    boxes, _ = mtcnn.detect(img)
    
    if boxes is None:
        print("❌ No se detectó ningún rostro en la imagen.")
    else:
        print(f"✓ Se detectaron {len(boxes)} rostro(s).")
        
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(img)
        
        # Procesar cada rostro detectado
        for idx, box in enumerate(boxes):
            # Coordenadas enteras de la caja
            x1, y1, x2, y2 = map(int, box)
            
            # Recortar el rostro usando PIL
            face_crop = img.crop((x1, y1, x2, y2))
            
            # Preprocesar
            face_tensor = preprocess_transform(face_crop).unsqueeze(0).to(DEVICE)
            
            # Extraer embedding del clasificador
            with torch.no_grad():
                embedding, _ = model(face_tensor)  # embedding ya viene L2-normalizado
            
            # Calcular similitud coseno con los centroides (producto punto ya que están normalizados)
            similarities = torch.matmul(embedding, centroids.t()).squeeze(0)  # Shape: [26]
            
            max_sim, pred_idx = torch.max(similarities, dim=0)
            max_sim = max_sim.item()
            pred_idx = pred_idx.item()
            
            # Aplicar filtro de umbral
            if max_sim >= UMBRAL_RECHAZO:
                name_pred = idx_to_label[pred_idx]
                label_text = f"{name_pred} ({max_sim:.2%})"
                color = 'lime'
            else:
                label_text = f"Desconocido ({max_sim:.2%})"
                color = 'red'
                
            print(f"Rostro {idx+1}: {label_text}")
            
            # Dibujar caja en la imagen original
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 10, label_text, color='white', weight='bold', bbox=dict(facecolor=color, alpha=0.8, edgecolor='none'))
            
        plt.axis('off')
        plt.show()